# 05 - Product Segmentation

Beyond association rules, we group products into data-driven
segments by clustering their *co-occurrence patterns* with K-means.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline


In [ ]:
from src.preprocessing import preprocess_dataset

raw, transactions, encoded = preprocess_dataset()
print(f'Transactions: {len(transactions):,}')
print(f'Unique products: {encoded.shape[1]:,}')

### Co-occurrence matrix

Every product is described by how often it is bought together with
every other product. `encoded.T @ encoded` gives those counts.

In [ ]:
from src.clustering import co_occurrence_matrix

cooc = co_occurrence_matrix(encoded)
cooc.iloc[:5, :5]

### Run K-means clustering

The number of clusters is chosen with the silhouette score.
Products with fewer than 5 baskets are skipped to reduce noise.

In [ ]:
from src.clustering import cluster_products

assignments, silhouette = cluster_products(encoded, min_frequency=5)
print(f'Clusters found: {assignments["cluster"].nunique()}')
print(f'Silhouette score: {silhouette:.3f}')
assignments.head(15)

In [ ]:
from src.clustering import summarize_clusters

summary = summarize_clusters(assignments)
for cluster_id, info in summary.items():
    print(f"Segment {cluster_id + 1} ({info['size']} products, {info['share']:.0%}):")
    print('   ', ', '.join(info['top_products'][:6]))

In [ ]:
from src.clustering import run_clustering

stats = run_clustering(encoded)
print('Plot saved to:', stats['paths']['clusters_plot'])

## Interpretation

* Products clustered together share purchase context: they tend to
  appear in the same baskets.
* The segments are learned from the data, so they can reveal
  categories a manual taxonomy would miss.
* Combine with the association rules for a complete picture:
  rules tell you *what goes together*, segments tell you *which
  products form families*.